# 🚀 Notebook 6: Full RAG Pipeline
### Putting it all together — your MBA knowledge base is ready

**WAT Reference:** `workflows/06_full_pipeline.md` | `tools/ask.py`

---

## What This Notebook Does

This is where all 5 previous notebooks come together into one system:

```
You ask:  "What case studies did I study about disruptive innovation?"
                              ↓
[1] Embed question  →  [0.041, -0.318, ...]  (1,536 numbers)
                              ↓
[2] Search vector store  →  Top 5 most similar chunks
   Score 0.91  Harvard_Disruptive_Innovation.pdf
   Score 0.87  Strategy_Notes_Clayton_Christensen.docx
   Score 0.84  BCG_Case_Study_Netflix.pptx
                              ↓
[3] Build prompt  →  "Use ONLY this context to answer..."
                     + chunk 1 text (with source)
                     + chunk 2 text (with source)
                     + chunk 3 text (with source)
                     + question
                              ↓
[4] GPT-4 answers  →  Grounded answer + source citations
```

## What You'll Do

1. Build and test the complete pipeline on sample files
2. See WITH vs WITHOUT RAG comparison
3. Test subject filtering
4. **Phase 2: Scale to all 1,250 files using ChromaDB**
5. See how this maps to your n8n workflow

**Time needed:** ~20 minutes | **Cost:** ~$0.01–$0.05

In [ ]:
!pip install openai -q

import os, json, math
import openai

# ─────────────────────────────────────────────────────────────────────
# NOTE: This cell mounts Google Drive, which only works in Google Colab.
# If you're running in the browser (via the website), this cell will
# not work — that's expected. To run the full pipeline with your own
# files, open this notebook in Google Colab.
# ─────────────────────────────────────────────────────────────────────
try:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    IN_COLAB = True
    print("✅ Google Drive mounted successfully!")
except Exception:
    IN_COLAB = False
    print("ℹ️  Google Drive is not available in this environment.")
    print("   To run this notebook with your MBA files, open it in Google Colab.")
    print("   You can still read through the cells to understand the full pipeline.")

PROJECT_ROOT    = '/content/drive/MyDrive/MBA_RAG'
VECTOR_STORE    = os.path.join(PROJECT_ROOT, '.tmp/vector_store.json')
EMBEDDING_MODEL = 'text-embedding-3-small'
CHAT_MODEL      = 'gpt-4o-mini'

# ─────────────────────────────────────────────────────────────────────
# Load API key — works in both environments:
#
#   On the website:  key is injected automatically when you click
#                    "Run in browser" and enter it in the panel above.
#
#   In Google Colab: key is loaded from Colab Secrets (🔑 in sidebar).
#                    Add secret name: OPENAI_API_KEY
# ─────────────────────────────────────────────────────────────────────
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', '')

if not OPENAI_API_KEY and IN_COLAB:
    try:
        OPENAI_API_KEY = userdata.get('OPENAI_API_KEY') or ''
        if OPENAI_API_KEY:
            os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
    except Exception:
        pass

if OPENAI_API_KEY:
    print(f"✅ API key loaded. Starts with: {OPENAI_API_KEY[:8]}...")
else:
    print("⚠️  API key not found.")
    print("   On the website:  click 'Run in browser' and enter your key first.")
    print("   In Google Colab: add it via the 🔑 Secrets panel in the sidebar.")

client = openai.OpenAI(api_key=OPENAI_API_KEY)

# Load vector store (only if Drive is available)
if IN_COLAB and os.path.exists(VECTOR_STORE):
    with open(VECTOR_STORE, 'r', encoding='utf-8') as f:
        chunks = json.load(f)
    print(f"✅ Loaded {len(chunks):,} chunks from vector_store.json")
    print(f"   Ready to answer questions about your MBA files!")
elif IN_COLAB:
    print(f"⚠️  vector_store.json not found at {VECTOR_STORE}")
    print("   Run Notebooks 2 → 3 → 4 first to generate it.")
    chunks = []
else:
    chunks = []

In [ ]:
# Helper functions (same as Notebooks 4 & 5 — combined here for convenience)

def cosine_similarity(vec_a, vec_b):
    dot = sum(a * b for a, b in zip(vec_a, vec_b))
    na  = math.sqrt(sum(a*a for a in vec_a))
    nb  = math.sqrt(sum(b*b for b in vec_b))
    return dot / (na * nb) if na and nb else 0.0

def embed(text):
    return client.embeddings.create(model=EMBEDDING_MODEL, input=text).data[0].embedding

def retrieve(query, subject=None, top_k=5):
    qv   = embed(query)
    pool = [c for c in chunks if c.get('subject','').lower() == subject.lower()] if subject else chunks
    if subject and not pool:
        pool = chunks
    scored = sorted(pool, key=lambda c: cosine_similarity(qv, c['embedding']), reverse=True)
    return [{**c, 'score': round(cosine_similarity(qv, c['embedding']), 4)} for c in scored[:top_k]]

print("✅ Helper functions ready.")

---
## Step 1: The RAG Prompt — The Core of the System

The prompt design is what separates a good RAG system from a hallucinating one.

Three critical instructions:
1. **"Use ONLY the context"** — prevents GPT-4 from using training knowledge
2. **"Cite source files"** — every answer is traceable
3. **"Say I don't know"** — honest rather than wrong

In [ ]:
def build_rag_prompt(question, retrieved_chunks):
    """
    Build the RAG prompt that combines:
    - Instructions for GPT-4 (anti-hallucination + source citation)
    - Retrieved chunks as context (the actual MBA content)
    - The user's question
    """
    context_parts = []
    for i, chunk in enumerate(retrieved_chunks, 1):
        # Use full path (source) so duplicate filenames in different subfolders are distinguishable
        display_source = chunk.get('source', chunk['filename'])
        context_parts.append(
            f"[Source {i}: {display_source} — {chunk['subject']}]\n"
            f"{chunk['text']}"
        )
    context = "\n\n".join(context_parts)

    return f"""You are an MBA knowledge assistant helping the user recall what they studied.

Use ONLY the information in the CONTEXT section below to answer the question.
If the answer is not found in the context, say: "I don't have that information in my MBA files."
Do NOT make up, infer, or add any information beyond what is in the context.
At the end, list the source file(s) you used on a line starting with "Sources:".

CONTEXT:
{'─' * 60}
{context}
{'─' * 60}

QUESTION: {question}

ANSWER:"""


def ask(question, subject=None, top_k=5, show_chunks=False):
    """Complete RAG pipeline for one question."""
    # Step 1: Retrieve
    results = retrieve(question, subject=subject, top_k=top_k)

    if show_chunks:
        print("Retrieved chunks:")
        for r in results:
            print(f"  {r['score']} | {r.get('source', r['filename'])} ({r['subject']})")
        print()

    # Step 2: Build prompt
    prompt = build_rag_prompt(question, results)

    # Step 3: Call GPT-4
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content


print("✅ Full RAG pipeline ready.")


---
## Step 2: Ask Your First Question

Let's test the full pipeline — with chunk visibility so you can see what GPT-4 is reading.

In [ ]:
# ─────────────────────────────────────────────────────────────
# ✏️  Change this to any question about your MBA files
# ─────────────────────────────────────────────────────────────
MY_QUESTION = "What frameworks did I learn for competitive analysis?"

print(f"❓ QUESTION: {MY_QUESTION}")
print()

answer = ask(MY_QUESTION, show_chunks=True)

print("💬 ANSWER:")
print("─" * 60)
print(answer)
print("─" * 60)

---
## Step 3: WITH vs WITHOUT RAG — Final Comparison

In [ ]:
def ask_without_rag(question):
    """Standard GPT-4 — no context from your files."""
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user",   "content": question}
        ]
    )
    return response.choices[0].message.content


comparison_q = "What specific case studies did I study in my MBA program?"

print("=" * 70)
print(f"QUESTION: {comparison_q}")
print("=" * 70)

print("\n🚫 WITHOUT RAG (GPT-4 guessing):")
print("─" * 70)
no_rag = ask_without_rag(comparison_q)
print(no_rag[:600] + "..." if len(no_rag) > 600 else no_rag)

print("\n✅ WITH RAG (GPT-4 reading your actual files):")
print("─" * 70)
with_rag = ask(comparison_q)
print(with_rag)

print("\n" + "=" * 70)
print("Without RAG: generic, possibly mentions cases you never studied")
print("With RAG:    cites actual files from your data/sample/ folder")
print("=" * 70)

---
## Step 4: Subject-Filtered Q&A

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ✏️  Ask a domain-specific question with subject filtering
#     Change SUBJECT to one of your actual folder names
# ─────────────────────────────────────────────────────────────────────────────
DOMAIN_QUESTION = "How is net present value used in investment decisions?"
SUBJECT_FILTER  = "Finance"   # ← Change to your actual folder name

print(f"❓ QUESTION: {DOMAIN_QUESTION}")
print(f"📁 Subject filter: {SUBJECT_FILTER}")
print()

domain_answer = ask(DOMAIN_QUESTION, subject=SUBJECT_FILTER, show_chunks=True)

print("💬 ANSWER:")
print("─" * 60)
print(domain_answer)
print("─" * 60)

---
## Step 5: Interactive Q&A Loop

Run this cell for a continuous Q&A session. Type your questions and press Enter. Type `quit` to stop.

In [ ]:
print("MBA KNOWLEDGE BASE Q&A")
print("=" * 60)
print("Ask anything about your MBA files.")
print("Add 'subject=Finance' (etc.) to filter by folder.")
print("Type 'quit' to exit.")
print()

while True:
    user_input = input("Your question: ").strip()
    if not user_input:
        continue
    if user_input.lower() == 'quit':
        print("Session ended.")
        break

    # Parse optional subject filter
    subj = None
    question = user_input
    if 'subject=' in user_input.lower():
        parts    = user_input.lower().split('subject=')
        subj     = user_input.split('subject=', 1)[1].strip()
        question = user_input.split('subject=')[0].strip()

    print()
    try:
        result = ask(question, subject=subj)
        print("─" * 60)
        print(result)
        print("─" * 60)
    except Exception as e:
        print(f"Error: {e}")
    print()

---
## Phase 2: Scale to All 1,250 Files with ChromaDB

The JSON file works well for hundreds of chunks. For 20,000+ chunks, searching becomes slow (comparing query against every chunk one-by-one takes seconds instead of milliseconds).

**ChromaDB** is a free local vector database that:
- Stores embeddings in an indexed structure → much faster search
- Supports native metadata filtering (`subject=Finance`)
- Runs entirely locally — no cloud, no extra cost

This section shows how to migrate your JSON store to ChromaDB.

In [ ]:
!pip install chromadb -q
print("✅ ChromaDB installed")

In [ ]:
import chromadb

CHROMA_PATH = os.path.join(PROJECT_ROOT, '.tmp/chroma_db')

# Create a ChromaDB client that stores data on Google Drive
chroma_client  = chromadb.PersistentClient(path=CHROMA_PATH)
collection     = chroma_client.get_or_create_collection(name='mba_knowledge')

print(f"ChromaDB collection: 'mba_knowledge'")
print(f"Storage location: {CHROMA_PATH}")
print()

existing_count = collection.count()
if existing_count > 0:
    print(f"ℹ️  Collection already has {existing_count} chunks — skipping migration.")
    print("   Delete .tmp/chroma_db/ to re-migrate.")
else:
    print(f"Migrating {len(chunks)} chunks from JSON to ChromaDB...")
    BATCH = 500
    for i in range(0, len(chunks), BATCH):
        batch = chunks[i : i + BATCH]
        collection.add(
            ids        = [str(c['chunk_id']) for c in batch],
            embeddings = [c['embedding']     for c in batch],
            documents  = [c['text']          for c in batch],
            # Store full source path so duplicate filenames across subfolders stay distinguishable
            metadatas  = [{
                'filename': c['filename'],
                'source':   c.get('source', c['filename']),
                'subject':  c.get('subject', '')
            } for c in batch],
        )
        print(f"  Migrated {min(i+BATCH, len(chunks))}/{len(chunks)}")

    print(f"\n✅ Migration complete! {collection.count()} chunks in ChromaDB.")


In [ ]:
def ask_chromadb(question, subject=None, top_k=5, show_chunks=False):
    """Full RAG pipeline using ChromaDB for retrieval."""

    # Embed question
    query_vector = embed(question)

    # Query ChromaDB — with optional metadata filter
    where = {'subject': subject} if subject else None
    results = collection.query(
        query_embeddings=[query_vector],
        n_results=top_k,
        where=where,
        include=['documents', 'metadatas', 'distances']
    )

    # Format retrieved chunks (ChromaDB returns distance, convert to similarity)
    retrieved = []
    for text, meta, dist in zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    ):
        retrieved.append({
            'text':     text,
            'filename': meta['filename'],
            'source':   meta.get('source', meta['filename']),  # full path for disambiguation
            'subject':  meta['subject'],
            'score':    round(1 - dist, 4),
        })

    if show_chunks:
        print("ChromaDB retrieved:")
        for r in retrieved:
            print(f"  {r['score']} | {r['source']} ({r['subject']})")
        print()

    # Build prompt and call GPT-4 (same as before)
    prompt   = build_rag_prompt(question, retrieved)
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content


# Test ChromaDB pipeline
print("Testing ChromaDB pipeline...")
print()
test_q  = "What did I study about competitive strategy?"
chroma_answer = ask_chromadb(test_q, show_chunks=True)
print("💬 ANSWER:")
print("─" * 60)
print(chroma_answer)
print("─" * 60)
print()
print("✅ ChromaDB works! Same answers, faster retrieval at scale.")


---
## The n8n Connection — What You Built Before

You already built a basic RAG in n8n. Every node in that workflow maps directly to what we just built in Python.

In [ ]:
n8n_mapping = """
┌─────────────────────────────────────────────────────────────────────┐
│              YOUR n8n WORKFLOW  ←→  THESE PYTHON TOOLS              │
├─────────────────────────────────┬───────────────────────────────────┤
│ n8n Node                        │ Python Tool                       │
├─────────────────────────────────┼───────────────────────────────────┤
│ Document Loader node            │ tools/parse_documents.py          │
│  (reads your PDF)               │  (reads PDF/DOCX/PPTX/XLSX)      │
├─────────────────────────────────┼───────────────────────────────────┤
│ Text Splitter node              │ tools/chunk_text.py               │
│  (splits into chunks)           │  (500-token chunks, 50 overlap)   │
├─────────────────────────────────┼───────────────────────────────────┤
│ Embeddings node                 │ tools/embed_chunks.py             │
│  (OpenAI Embeddings)            │  (text-embedding-3-small)         │
├─────────────────────────────────┼───────────────────────────────────┤
│ Vector Store node               │ .tmp/vector_store.json            │
│  (stores vectors)               │  OR .tmp/chroma_db/              │
├─────────────────────────────────┼───────────────────────────────────┤
│ Vector Store Retriever node     │ tools/retrieve.py                 │
│  (finds relevant chunks)        │  (cosine similarity search)       │
├─────────────────────────────────┼───────────────────────────────────┤
│ AI Chain / Agent node           │ tools/ask.py                      │
│  (sends to LLM + gets answer)   │  (builds prompt + calls GPT-4)    │
└─────────────────────────────────┴───────────────────────────────────┘

In n8n: you connected nodes visually
In Python: you connected functions — same logic, full visibility
"""

print(n8n_mapping)

---
## Phase 2: Running on All 1,250 Files

When you're ready to process your full MBA corpus:

1. Upload all your files to Google Drive at `MBA_RAG/data/full/` keeping the 30-folder structure
2. In Notebook 2 (`02_parse_documents.ipynb`), change:
   ```python
   DATA_FOLDER = os.path.join(PROJECT_ROOT, 'data/sample')
   # → change to:
   DATA_FOLDER = os.path.join(PROJECT_ROOT, 'data/full')
   ```
3. Re-run Notebooks 2, 3, 4 in sequence (one click each)
4. Come back here and run the ChromaDB migration cell
5. Use `ask_chromadb()` for all future questions

**Time:** ~15-30 minutes for 1,250 files  
**Cost:** ~$0.20–$0.30 (one-time embedding)

After that, every question costs ~$0.001 and answers in ~2 seconds.

---
## 🎉 Congratulations — You Built a RAG System!

### What you built across 6 notebooks:

| Step | What it does | Tool |
|------|-------------|------|
| Parse | Extract text from PDF/DOCX/PPTX/XLSX | `tools/parse_documents.py` |
| Chunk | Split into 500-token pieces with overlap | `tools/chunk_text.py` |
| Embed | Convert text to 1,536-number vectors | `tools/embed_chunks.py` |
| Store | Save to JSON or ChromaDB | `.tmp/` |
| Retrieve | Find top-K by cosine similarity | `tools/retrieve.py` |
| Answer | Build RAG prompt + call GPT-4 | `tools/ask.py` |

### What makes it work:
- **Embeddings** capture meaning as numbers — similar content clusters together
- **Cosine similarity** measures how close two meanings are
- **Anti-hallucination prompt** forces GPT-4 to use only your documents
- **Subject tagging** enables precision retrieval by folder

### Your knowledge base can now answer:
- *"What case studies did I study about competitive advantage?"*
- *"Summarize my notes on the Capital Asset Pricing Model"*
- *"What frameworks did I learn in my Marketing classes?"*
- *"What was the main argument in the Blue Ocean Strategy case?"*

All grounded in YOUR actual files, with source citations. ✅

---
**Update README.md — check off ALL notebooks ✅ and start Phase 2!**